## 📦 Install dependencies
Run this cell once.

In [ ]:
%pip install -q pymupdf pymupdf4llm python-docx chromadb ollama reportlab
# sample file for testing
!python make_sample.py

# 🛡️ SLM Powered Regulatory
A 100% local RAG system that compares an organization's documents (PDF / Word) against **ISO/IEC 27001:2022**
and reports what is ✅ compliant, 🟡 partially compliant, ❌ non-compliant, or ⚪ has no evidence, together with the supporting evidence and a recommended fix.

**Full pipeline:**
`Organization file → extract & clean → chunks → vector DB → RAG → LLM auditor → report + dashboard`

Before you start, run in a terminal:
```bash
ollama pull llama3:8b     # the LLM (use qwen2.5:7b on a weaker machine)
ollama pull bge-m3          # the embedding model (multilingual)
```

## 0️⃣ Configuration
All the important settings live in one place. Change the model name here only.
- `MIN_RELEVANCE`: the minimum similarity accepted. Any passage below it is ignored, which prevents answers built on unrelated text.
- `TOP_K`: how many passages are retrieved per question or requirement.

In [ ]:
import os, re, json, unicodedata
from pathlib import Path
from collections import Counter
from datetime import date
import pymupdf4llm
import fitz
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph

import chromadb
import ollama
import pandas as pd

LLM_MODEL = "llama3:8b"
EMBED_MODEL = "bge-m3"
DB_PATH = "./vector_db"
COMPANY_COL = "company_docs"
ISO_COL = "iso27001"
REQ_FILE = r"C:\Users\sasap\OneDrive\Desktop\New folder\slm-regulatory\data\iso-27001-annex-a.csv"

# Explicit mapping for the supplied CSV (avoids confusing Theme with Control ID).
CSV_COLUMNS = {"id": "Control", "title": "Theme", "requirement": "Summary", "evidence": "Evidence"}

MAX_CHARS = 1500
CHUNK_OVERLAP = 180
TOP_K = 8
MIN_RELEVANCE = 0.32
REL_CUTOFF = 0.62
NUM_CTX = 8192
OCR_ENABLED = True       # OCR is attempted only for pages with no extractable text.
OCR_LANG = "eng"         # Add Arabic traineddata and use "eng+ara" for bilingual scans.

## 1️⃣ Load files (PDF / Word)
- **PDF:** `pymupdf4llm` converts each page to Markdown, keeping headings, lists and **tables** (a table becomes a Markdown table, so numbers stay in the right cell).
- **Word:** paragraphs and tables are read in their original order and converted to Markdown as well.

The output is always a list of `(page number, Markdown text)`.

In [ ]:

def table_to_md(tbl):
    rows = [[c.text.strip().replace("\n", " ") for c in r.cells] for r in tbl.rows]
    if not rows:
        return ""
    width = max(len(r) for r in rows)
    rows = [r + [""] * (width - len(r)) for r in rows]
    md = ["| " + " | ".join(rows[0]) + " |", "|" + "---|" * width]
    md += ["| " + " | ".join(r) + " |" for r in rows[1:]]
    return "\n".join(md)


def read_docx(path):
    doc = Document(path)
    out = []
    for block in doc.element.body.iterchildren():
        if block.tag.endswith("}p"):
            p = Paragraph(block, doc)
            text = p.text.strip()
            if not text:
                continue
            style = p.style.name if p.style is not None else ""
            if style.startswith("Heading"):
                level = "".join(ch for ch in style if ch.isdigit()) or "1"
                out.append("#" * min(int(level), 6) + " " + text)
            elif "List" in style:
                out.append("- " + text)
            else:
                out.append(text)
        elif block.tag.endswith("}tbl"):
            out.append(table_to_md(Table(block, doc)))
    return [(0, "\n\n".join(out))]


def _ocr_pdf_page(page):
    """Optional OCR fallback; requires pytesseract, Pillow and a Tesseract language pack."""
    if not OCR_ENABLED:
        return ""
    try:
        import pytesseract
        from PIL import Image
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
        import io
        return pytesseract.image_to_string(Image.open(io.BytesIO(pix.tobytes("png"))), lang=OCR_LANG).strip()
    except Exception as exc:
        print(f"OCR skipped on page {page.number + 1}: {exc}")
        return ""


def read_pdf(path):
    """Extract page-wise Markdown first; fall back to PyMuPDF text and optional OCR per page."""
    try:
        pages = pymupdf4llm.to_markdown(path, page_chunks=True)
        result = [(i, str(pg.get("text", ""))) for i, pg in enumerate(pages, 1)]
    except Exception as exc:
        print(f"Markdown extraction fallback: {exc}")
        result = []
    doc = fitz.open(path)
    if not result or not any(t.strip() for _, t in result):
        result = [(i + 1, page.get_text("text")) for i, page in enumerate(doc)]
        
    fixed = []
    for i, page in enumerate(doc, 1):
        current = result[i - 1][1] if i <= len(result) else ""
        if len(current.strip()) < 20:
            fallback = page.get_text("text").strip()
            current = fallback if len(fallback) > len(current) else current
            if len(current.strip()) < 20:
                current = _ocr_pdf_page(page) or current
        fixed.append((i, current))
    doc.close()
    return fixed


def read_file(path):
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        return read_pdf(path)
    if ext == ".docx":
        return read_docx(path)
    raise ValueError(f"Unsupported file type: {ext}")

d:\Anaconda\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [3]:
DEMO = r"C:\Users\sasap\OneDrive\Desktop\New folder\slm-regulatory\data\northstar_company_security_governance_dossier.pdf"
raw_pages = read_file(DEMO)
print(f"Extracted {len(raw_pages)} pages")
print(raw_pages[0][1][:900])


=== Document parser messages ===
Extracted 21 pages
# **NORTHSTAR DIGITAL SERVICES** 

Information Security & Governance Dossier 

Fictional Organization Profile for Regulatory Compliance Assessment 

|**Field**|**Value**|
|---|---|
|Document ID|NDS-ISMS-DOS-001|
|Version|1.4|
|Issue date|15 October 2026|
|Document owner|Information Security Office|
|Approval authority|Chief Operating Officer|
|Classification|Internal - Controlled|
|Assessment purpose|Evidence-based comparison against ISO/IEC 27001 requirements and Annex A controls|



#### **TRAINING / DEMONSTRATION DATA — NOT A REAL COMPANY RECORD** 

This dossier describes a fictional organization. It contains deliberately varied evidence quality, dates, metrics, exceptions, and control implementation details to test document extraction, retrieval, citation, and compliance reasoning. It must not be treated as a real certification record or legal opinion. 




## 2️⃣ Clean and chunk
Cleaning normalizes Unicode, whitespace and line-wrapped words while conservatively removing repeated page furniture.

PDF extraction is page-aware and tries Markdown/table extraction first, then PyMuPDF text, with optional OCR for pages that have no usable text. OCR requires Tesseract and its language data; it is not silently assumed to be installed.

Chunking preserves section headings, page/source metadata and table headers. Long text is split at sentence boundaries with a small overlap.


In [4]:
def clean_text(t):
    t = unicodedata.normalize("NFKC", str(t)).replace("\x00", "")
    t = t.replace("\u00ad", "")
    t = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", t)  # join line-wrapped hyphenation
    t = re.sub(r"[ \t\u00a0]+", " ", t)
    t = re.sub(r" *\| *", " | ", t)
    t = re.sub(r"\n[ \t]*\n{2,}", "\n\n", t)
    return t.strip()


def remove_repeated_lines(pages):
    """Remove only short, exact repeated page furniture; preserve repeated evidence statements."""
    if len(pages) < 5:
        return pages
    counts = Counter()
    for _, text in pages:
        seen = set()
        for line in text.splitlines():
            line = re.sub(r"\s+", " ", line).strip()
            if 8 <= len(line) <= 90 and not line.startswith("|"):
                seen.add(line)
        counts.update(seen)
    repeated = {line for line, count in counts.items() if count / len(pages) >= 0.82}
    if not repeated:
        return pages
    out = []
    for page, text in pages:
        lines = [line for line in text.splitlines()
                 if re.sub(r"\s+", " ", line).strip() not in repeated]
        out.append((page, "\n".join(lines)))
    return out


def split_table(tbl):
    lines = tbl.splitlines()
    if len(lines) <= 2:
        return [tbl]
    head = "\n".join(lines[:2])
    parts, cur = [], []
    for row in lines[2:]:
        candidate = "\n".join(cur + [row])
        if cur and len(head) + len(candidate) > MAX_CHARS:
            parts.append(head + "\n" + "\n".join(cur))
            cur = []
        cur.append(row)
    if cur:
        parts.append(head + "\n" + "\n".join(cur))
    return parts


def split_long(text):
    sentences = re.split(r"(?<=[.!?؟])\s+|\n(?=\S)", text)
    parts, cur = [], ""
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if cur and len(cur) + len(sentence) + 1 > MAX_CHARS:
            parts.append(cur.strip())
            # Keep a small tail as context, without duplicating an entire paragraph.
            tail = cur[-CHUNK_OVERLAP:].strip()
            cur = (tail + " " if tail else "") + sentence
        else:
            cur = (cur + " " + sentence).strip()
    if cur:
        parts.append(cur.strip())
    return parts


def chunk_pages(pages, source):
    """Heading-aware, page-aware chunks; table headers repeat when a table is split."""
    chunks, heading, buf, buf_page = [], "", "", 0
    def flush():
        nonlocal buf
        if buf.strip():
            prefix = f"[Section: {heading}]\n" if heading else ""
            chunks.append({"text": prefix + buf.strip(), "source": source,
                           "page": buf_page, "heading": heading})
        buf = ""
    for page, text in pages:
        text = clean_text(text)
        for block in re.split(r"\n\s*\n", text):
            block = block.strip()
            if not block:
                continue
            heading_match = re.match(r"^(#{1,6})\s+(.+)$", block)
            if heading_match:
                flush()
                heading = heading_match.group(2).strip()[:180]
                block = block[len(heading_match.group(0)):].strip()
                if not block:
                    continue
            is_table = block.startswith("|")
            if len(block) > MAX_CHARS:
                flush()
                pieces = split_table(block) if is_table else split_long(block)
                for piece in pieces:
                    buf, buf_page = piece, page
                    flush()
                continue
            if buf and len(buf) + len(block) > MAX_CHARS:
                flush()
            if not buf:
                buf_page = page
            buf += block + "\n\n"
    flush()
    return chunks

In [5]:
pages = remove_repeated_lines([(n, clean_text(t)) for n, t in read_file(DEMO)])
chunks = chunk_pages(pages, os.path.basename(DEMO))
print(len(chunks), "chunks")
for c in chunks[:3]:
    print("-" * 50)
    print(c["text"])


=== Document parser messages ===
72 chunks
--------------------------------------------------
[Section: **NORTHSTAR DIGITAL SERVICES**]
Information Security & Governance Dossier

Fictional Organization Profile for Regulatory Compliance Assessment

| **Field** | **Value** | 
 | --- | --- | 
 | Document ID | NDS-ISMS-DOS-001 | 
 | Version | 1.4 | 
 | Issue date | 15 October 2026 | 
 | Document owner | Information Security Office | 
 | Approval authority | Chief Operating Officer | 
 | Classification | Internal - Controlled | 
 | Assessment purpose | Evidence-based comparison against ISO/IEC 27001 requirements and Annex A controls |
--------------------------------------------------
[Section: **TRAINING / DEMONSTRATION DATA — NOT A REAL COMPANY RECORD**]
This dossier describes a fictional organization. It contains deliberately varied evidence quality, dates, metrics, exceptions, and control implementation details to test document extraction, retrieval, citation, and compliance reasoning.

## 3️⃣ Embeddings and vector database
- **Embeddings:** the `bge-m3` model served by Ollama (local, multilingual).
- **Vector DB:** `ChromaDB`, stored on disk in the `vector_db` folder, so data survives restarts.
- Each chunk is stored with its **file, page and section** so answers can cite their source.

**Why numbers come back correctly:** embeddings alone are weak with numbers ("12 characters" and "8 characters" look similar to them), so retrieval is **hybrid**:
70% semantic similarity + 30% exact word/number overlap. Tables are also kept whole, so a number always stays next to its context.

The ISO requirements are stored in a separate collection so we can question them or compare against them.

In [6]:
client = chromadb.PersistentClient(path=DB_PATH)


def get_collection(name):
    return client.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})


def reset_collection(name):
    try:
        client.delete_collection(name)
    except Exception:
        pass


def embed(texts, batch=32):
    vectors = []
    for i in range(0, len(texts), batch):
        vectors += ollama.embed(model=EMBED_MODEL, input=texts[i:i + batch])["embeddings"]
    return vectors


def index_chunks(chunks, collection_name, batch=32):
    col = get_collection(collection_name)
    for start in range(0, len(chunks), batch):
        part = chunks[start:start + batch]
        col.upsert(
            ids=[f"{c['source']}::{start + j}" for j, c in enumerate(part)],
            documents=[c["text"] for c in part],
            embeddings=embed([c["text"] for c in part]),
            metadatas=[{"source": c["source"], "page": c.get("page", 0), "heading": c.get("heading", ""),
                       "location": c.get("location", "")} for c in part],
        )
    return len(chunks)


def ingest_file(path, collection_name=COMPANY_COL):
    """Full ingestion: read -> clean -> chunk -> embed -> store."""
    pages = remove_repeated_lines([(n, clean_text(t)) for n, t in read_file(path)])
    chunks = chunk_pages(pages, os.path.basename(path))
    return index_chunks(chunks, collection_name)



_ALIASES = {
    "id": ["id", "control", "clause", "control id", "controlid", "ref", "reference", "number", "no", "control no", "control number"],
    "requirement": ["requirement", "requirements", "description", "text", "control text", "statement", "details", "objective", "control description", "summary"],
    "title": ["title", "name", "control name", "heading", "topic", "domain", "category", "theme"],
    "evidence": ["evidence", "expected evidence", "audit evidence", "evidence required", "artifacts"],
    "stage": ["stage", "maturity", "level"]
}

def _n(s):
    return re.sub(r"[\s_\-]+", " ", str(s).strip().lower())


def load_requirements(path=None):
    """Load ISO rows while retaining Control, Theme, Stage, Summary and Evidence."""
    path = Path(path or REQ_FILE)
    if path.suffix.lower() == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        return data if isinstance(data, list) else data.get("requirements", [])
    if path.suffix.lower() != ".csv":
        raise ValueError("Requirements file must be CSV or JSON")
    df = None
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            df = pd.read_csv(path, sep=None, engine="python", encoding=enc, dtype=str).fillna("")
            break
        except UnicodeDecodeError:
            continue
    if df is None:
        raise ValueError(f"Could not decode requirements CSV: {path}")
    mapping = dict(CSV_COLUMNS)
    normalized = {_n(c): c for c in df.columns}
    for key in ("id", "requirement", "title", "evidence", "stage"):
        if key in mapping:
            continue
        for alias in _ALIASES[key]:
            if alias in normalized:
                mapping[key] = normalized[alias]
                break
    missing = [k for k in ("id", "requirement") if k not in mapping]
    if missing:
        raise ValueError(f"Missing required CSV mapping(s) {missing}; columns found: {list(df.columns)}")
    reqs = []
    for row_num, row in df.iterrows():
        rid = str(row.get(mapping.get("id", ""), "")).strip()
        requirement = str(row.get(mapping.get("requirement", ""), "")).strip()
        if not rid or not requirement:
            continue
        theme = str(row.get(mapping.get("title", ""), "")).strip()
        stage = str(row.get(mapping.get("stage", ""), "")).strip()
        evidence = str(row.get(mapping.get("evidence", ""), "")).strip()
        title = " — ".join(x for x in (theme, stage) if x) or rid
        reqs.append({"id": rid, "title": title, "theme": theme, "stage": stage,
                     "requirement": requirement, "evidence": evidence, "source_row": int(row_num) + 2})
    if not reqs:
        raise ValueError("No valid control rows were loaded from the requirements file")
    return reqs


def ensure_iso_kb(force=False):
    col = get_collection(ISO_COL)
    label = f"ISO requirements ({Path(REQ_FILE).name})"
    if force:
        try:
            col.delete(where={"source": label})
        except Exception:
            pass
    elif col.count():
        existing = col.get(where={"source": label}, limit=1)
        if existing.get("ids"):
            return
    reqs = load_requirements()
    chunks = [{"text": f"{r['id']} {r['title']}: {r['requirement']} Expected evidence examples: {r['evidence']}",
               "source": label, "page": 0, "heading": r["id"],
               "location": f"CSV row {r.get('source_row', '?')}"} for r in reqs]
    index_chunks(chunks, ISO_COL)

In [7]:
reset_collection(COMPANY_COL)
print("company chunks:", ingest_file(DEMO, COMPANY_COL))
ensure_iso_kb(force=True)
print("company chunks:", get_collection(COMPANY_COL).count())
print("ISO controls   :", get_collection(ISO_COL).count())


=== Document parser messages ===
company chunks: 72
company chunks: 72
ISO controls   : 93


## 4️⃣ RAG: retrieve from the documents only
The LLM sees **nothing except** what we retrieve from the vector DB. Hallucination is limited in three layers:
1. **Similarity filter:** if no passage is close enough, there is no answer at all.
2. **Strict prompt:** "answer from the CONTEXT only; otherwise say Not found."
3. **Temperature = 0**, so answers are deterministic.

Every passage gets a tag: `[C1]` for company files and `[I1]` for ISO.

In [8]:
_STOPWORDS = {"the", "and", "for", "with", "from", "that", "this", "are", "was", "were", "into", "must", "should", "shall", "each", "their", "its", "has", "have", "been", "being", "all", "any", "not", "only", "or", "of", "to", "in", "on", "by", "a", "an"}

def _tokens(text):
    return {x for x in re.findall(r"[a-zA-Z0-9]+(?:[._/-][a-zA-Z0-9]+)*", str(text).lower())
            if len(x) > 1 and x not in _STOPWORDS}

def lexical_score(query, text):
    q, t = _tokens(query), _tokens(text)
    if not q:
        return 0.0
    overlap = len(q & t) / len(q)
    qnums = {x for x in q if any(ch.isdigit() for ch in x)}
    tnums = {x for x in t if any(ch.isdigit() for ch in x)}
    number_match = len(qnums & tnums) / len(qnums) if qnums else 0.0
    return min(1.0, 0.82 * overlap + 0.18 * number_match) if qnums else overlap


def retrieve(query, collection_name, k=TOP_K):
    col = get_collection(collection_name)
    if col.count() == 0:
        return []
    candidate_n = min(max(k * 5, 20), col.count())
    res = col.query(query_embeddings=[embed([query])[0]], n_results=candidate_n)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        sim = max(0.0, 1.0 - float(dist))
        lex = lexical_score(query, doc)
        score = 0.68 * sim + 0.32 * lex
        if sim < MIN_RELEVANCE and lex < 0.12:
            continue
        hits.append({"text": doc, **meta, "semantic": round(sim, 3),
                     "lexical": round(lex, 3), "score": round(score, 3)})
    hits.sort(key=lambda h: h["score"], reverse=True)
    selected = []
    seen = set()
    for hit in hits:
        identity = (hit.get("source"), hit.get("page"), hit.get("heading"))
        if identity in seen:
            continue
        if selected and hit["score"] < selected[0]["score"] * REL_CUTOFF:
            continue
        selected.append(hit)
        seen.add(identity)
        if len(selected) >= k:
            break
    return selected


def build_context(hits, tag):
    return "\n\n".join(
        f"[{tag}{i}] ({h['source']}, {h.get('location') or ('page ' + str(h['page']) if h['page'] else 'location n/a')}, section: {h['heading'] or '-'})\n{h['text']}"
        for i, h in enumerate(hits, 1))

In [9]:
for h in retrieve("minimum password length", COMPANY_COL):
    print(h["score"], h["source"], "p." + str(h["page"]), "|", h["text"][:150].replace("\n", " "))

0.788 northstar_company_security_governance_dossier.pdf p.8 | [Section: **5.6 Password and Session Settings**] | **Setting** | **Corporate baseline** | **Current observation** |   | --- | --- | --- |   | Password


## 5️⃣ The LLM prompts
There are **two prompts**:

**(a) Q&A prompt:** any question is answered from the company files or ISO, with source citations.

**(b) Auditor prompt:** the model acts as a certified ISO lead auditor. It receives one requirement plus the company evidence and returns JSON containing:
`status` (compliant / partial / non_compliant / no_evidence), `reasoning`, `evidence` (word-for-word quotes), `gaps` and `recommendation`.

**Extra safety:** the model never writes quotes. It only returns the *tags* of the passages it relied on (e.g. `C1`), and the code attaches the real excerpt from that passage. So evidence can never be invented. If the model cites nothing, the status can't be `compliant`, and if there are no relevant passages the status is `no_evidence`.

**Calibration:** the auditor prompt defines exactly when to use each status, includes four short examples, and treats the CSV `Evidence` column as *examples of proof*, not a checklist.

In [10]:
SYSTEM_QA = """You are a strict ISO 27001 compliance assistant.
Answer ONLY from supplied context. Cite each factual claim with [C#] or [I#].
Do not infer implementation from policy wording alone. Preserve dates, counts and scope exactly.
If context does not support an answer, say: Not found in the provided documents.
Always answer in English."""

SYSTEM_JUDGE = """You are an evidence-based ISO/IEC 27001:2022 control assessor.
Assess ONE control using ONLY the supplied company passages. Do not infer facts or implementation.
The control summary defines the outcome to assess; the Evidence field is illustrative evidence guidance, NOT a checklist.
Distinguish documented intent (policy) from operational proof (records, samples, configuration, metrics, completed activities).
Assess scope, coverage, date/currentness, owner, sample/population, exceptions and operational results where available.
Explicitly identify contradictory evidence and do not silently resolve it. Prefer specific, dated operational records over generic statements, but report unresolved conflicts.
Statuses:
compliant: relevant evidence demonstrates the requirement is implemented and operating with adequate coverage; no material unresolved gap is shown.
partial: meaningful implementation exists, but material scope/coverage/operation/assurance gaps or unresolved contradictory evidence remain.
non_compliant: relevant evidence explicitly demonstrates failure, absence, or a practice contrary to the requirement. Do not use this merely because a document is silent.
no_evidence: no retrieved passage substantively addresses the requirement. Similar words alone are not evidence.
A policy alone normally cannot prove operating effectiveness when the requirement concerns operation. A missing record is an evidence gap, not proof that an activity never occurred.
Use only valid passage tags in supporting. Empty supporting is allowed only for no_evidence.
Return JSON only: {"status":"compliant|partial|non_compliant|no_evidence","reasoning":"1-3 fact-based sentences","supporting":["C1"],"gaps":["specific gap"],"contradictions":["specific conflict between sources, or empty"],"recommendation":"specific action"}.
Do not invent quotes. Do not use outside knowledge."""

STATUSES = ["compliant", "partial", "non_compliant", "no_evidence"]
STATUS_LABEL = {"compliant": "Compliant", "partial": "Partially compliant",
                "non_compliant": "Non-compliant", "no_evidence": "No evidence found"}
SCORE = {"compliant": 1.0, "partial": 0.5, "non_compliant": 0.0, "no_evidence": None}


def chat(system, user, as_json=False):
    kwargs = {"format": "json"} if as_json else {}
    resp = ollama.chat(model=LLM_MODEL, messages=[{"role": "system", "content": system},
                                                  {"role": "user", "content": user}],
                       options={"temperature": 0, "num_ctx": NUM_CTX}, **kwargs)
    return resp["message"]["content"]


def ask(question):
    company, iso = retrieve(question, COMPANY_COL), retrieve(question, ISO_COL)
    if not company and not iso:
        return "Not found in the provided documents.", []
    context = build_context(company, "C") + "\n\n" + build_context(iso, "I")
    return chat(SYSTEM_QA, f"CONTEXT:\n{context}\n\nQUESTION: {question}"), company + iso

In [11]:
answer, used = ask("What is the minimum password length and how often are backups taken?")
print(answer)

Based on the provided documents, the minimum password length is 14 characters for local accounts, as stated in [C1] (northstar_company_security_governance_dossier.pdf, page 8, section: **5.6 Password and Session Settings**).

Regarding backups, the frequency of backups varies depending on the workload. For production databases, backups are taken daily with point-in-time recovery enabled. For critical configuration repositories, backups are taken every 24 hours. The retention schedule specifies 35 daily recovery points and 12 monthly archives for production databases.


## 6️⃣ Compliance audit
Each supplied Annex A row is assessed independently against retrieved company passages. The auditor must cite retrieved passage tags; code validates those citations and attaches excerpts directly from the source text. Invalid or absent citations cannot support a positive status.

Score: compliant = 1, partial = 0.5, non-compliant = 0. `NO EVIDENCE FOUND` is excluded from the score denominator and reported separately. Assessment coverage is reported alongside the score so missing documents cannot silently depress or inflate the score.

Applicability still needs confirmation from the organization's Statement of Applicability (SoA); this CSV alone does not establish applicability.


In [12]:
def best_excerpt(passage, query, max_chars=520):
    """Return a source-grounded sentence/row; never ask the LLM to generate evidence text."""
    parts = [p.strip() for p in re.split(r"(?<=[.!?؟])\s+|\n", passage)
             if len(p.strip()) > 20 and not p.strip().startswith("|---")]
    if not parts:
        return passage[:max_chars]
    ranked = sorted(parts, key=lambda x: lexical_score(query, x), reverse=True)
    return ranked[0][:max_chars]


def _parse_judge_json(raw):
    try:
        data = json.loads(raw)
    except Exception:
        match = re.search(r"\{.*\}", raw, re.S)
        try:
            data = json.loads(match.group(0)) if match else {}
        except Exception:
            data = {}
    return data if isinstance(data, dict) else {}


def check_requirement(req):
    query = f"{req['id']} {req['title']} {req['requirement']} {req.get('evidence', '')}"
    hits = retrieve(query, COMPANY_COL)
    context = build_context(hits, "C") or "(no relevant passage was retrieved)"
    if not hits:
        return {"id": req["id"], "title": req["title"], "requirement": req["requirement"],
                "status": "no_evidence", "reasoning": "No substantively relevant company evidence was retrieved.",
                "evidence": [], "gaps": ["Evidence was not found in the supplied company documents."],
                "recommendation": "Request the relevant policy, operational record, configuration export or owner confirmation."}

    prompt = (f"CONTROL {req['id']} — {req['title']}\nREQUIREMENT: {req['requirement']}\n"
              + (f"EXPECTED EVIDENCE EXAMPLES (not a checklist): {req.get('evidence', '')}\n" if req.get("evidence") else "")
              + f"\nCOMPANY PASSAGES:\n{context}\n\nReturn only the specified JSON. Cite only the C-tags above.")
    data = _parse_judge_json(chat(SYSTEM_JUDGE, prompt, as_json=True))
    status = data.get("status") if data.get("status") in STATUSES else "no_evidence"
    raw_support = data.get("supporting", [])
    if not isinstance(raw_support, list):
        raw_support = []
    cited = []
    for item in raw_support:
        match = re.fullmatch(r"\s*C(\d+)\s*", str(item))
        if match:
            idx = int(match.group(1))
            if 1 <= idx <= len(hits) and idx not in cited:
                cited.append(idx)
    # Citation validation: only cited, retrieved passages are surfaced as verified evidence.
    evidence = []
    for idx in cited:
        hit = hits[idx - 1]
        evidence.append({"ref": f"{hit['source']} p.{hit['page'] or '-'}",
                         "quote": best_excerpt(hit["text"], req["requirement"]),
                         "verified": True, "retrieval_score": hit["score"], "tag": f"C{idx}"})
    reasoning = str(data.get("reasoning", "")).strip()
    contradictions = data.get("contradictions", [])
    if not isinstance(contradictions, list):
        contradictions = [str(contradictions)] if contradictions else []
    contradictions = [str(x).strip() for x in contradictions if str(x).strip()]
    gaps = data.get("gaps", [])
    if not isinstance(gaps, list):
        gaps = [str(gaps)] if gaps else []
    gaps = [str(g).strip() for g in gaps if str(g).strip()]
    recommendation = str(data.get("recommendation", "")).strip()

    # No valid source citation means the LLM judgment cannot be accepted as proven.
    if status != "no_evidence" and not evidence:
        status = "no_evidence"
        reasoning = "The model did not provide a valid citation to a retrieved company passage; the assessment is therefore unsubstantiated."
        gaps = ["No validated supporting passage was cited."]
        recommendation = "Obtain and assess source records that directly address this control."
    if status == "compliant" and gaps:
        status = "partial"  # internally inconsistent output is conservatively downgraded
    if status == "no_evidence":
        evidence = []
        if not gaps:
            gaps = ["No sufficient evidence was identified in the supplied documents."]
    return {"id": req["id"], "title": req["title"], "requirement": req["requirement"],
            "status": status, "reasoning": reasoning or "The model returned no usable assessment reasoning.",
            "evidence": evidence, "gaps": gaps, "contradictions": contradictions, "recommendation": recommendation,
            "retrieved_passages": len(hits), "validated_citations": len(evidence)}


def run_compliance(progress=None, limit=None):
    reqs = load_requirements()
    if limit is not None:
        reqs = reqs[:limit]
    results = []
    for i, req in enumerate(reqs, 1):
        result = check_requirement(req)
        results.append(result)
        if progress:
            progress(i, len(reqs), req)
    return results


def summarize(results):
    counts = Counter(r["status"] for r in results)
    total = len(results)
    assessed = [r for r in results if r["status"] != "no_evidence"]
    # Evidence coverage is reported separately; no-evidence controls are excluded from score denominator.
    score = (sum(SCORE[r["status"]] for r in assessed) / len(assessed) * 100) if assessed else None
    return {"total": total, "assessed": len(assessed), "unassessed": total - len(assessed),
            "coverage": round(len(assessed) / total * 100, 1) if total else 0,
            "score": round(score, 1) if score is not None else None,
            **{s: counts.get(s, 0) for s in STATUSES}}

In [13]:
results = run_compliance(lambda i, n, r: print(f"{i}/{n}  {r['id']}"))
summary = summarize(results)
print(summary)
for r in results[:3]:
    print(r["id"], r["status"], "|", r["recommendation"][:120])

1/93  A.5.1
2/93  A.5.2
3/93  A.5.3
4/93  A.5.4
5/93  A.5.5
6/93  A.5.6
7/93  A.5.7
8/93  A.5.8
9/93  A.5.9
10/93  A.5.10
11/93  A.5.11
12/93  A.5.12
13/93  A.5.13
14/93  A.5.14
15/93  A.5.15
16/93  A.5.16
17/93  A.5.17
18/93  A.5.18
19/93  A.5.19
20/93  A.5.20
21/93  A.5.21
22/93  A.5.22
23/93  A.5.23
24/93  A.5.24
25/93  A.5.25
26/93  A.5.26
27/93  A.5.27
28/93  A.5.28
29/93  A.5.29
30/93  A.5.30
31/93  A.5.31
32/93  A.5.32
33/93  A.5.33
34/93  A.5.34
35/93  A.5.35
36/93  A.5.36
37/93  A.5.37
38/93  A.6.1
39/93  A.6.2
40/93  A.6.3
41/93  A.6.4
42/93  A.6.5
43/93  A.6.6
44/93  A.6.7
45/93  A.6.8
46/93  A.7.1
47/93  A.7.2
48/93  A.7.3
49/93  A.7.4
50/93  A.7.5
51/93  A.7.6
52/93  A.7.7
53/93  A.7.8
54/93  A.7.9
55/93  A.7.10
56/93  A.7.11
57/93  A.7.12
58/93  A.7.13
59/93  A.7.14
60/93  A.8.1
61/93  A.8.2
62/93  A.8.3
63/93  A.8.4
64/93  A.8.5
65/93  A.8.6
66/93  A.8.7
67/93  A.8.8
68/93  A.8.9
69/93  A.8.10
70/93  A.8.11
71/93  A.8.12
72/93  A.8.13
73/93  A.8.14
74/93  A.8.15
75/93  A

## 7️⃣ PDF report
Built with `reportlab`. It contains the summary and compliance score, then the requirements grouped as non-compliant → partial → no evidence → compliant, each with its evidence, gaps and recommended fix.

> The report is in English because the default reportlab font does not support Arabic.

In [14]:
def build_pdf(results, summary, path, company="Organization"):
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, KeepTogether
    from xml.sax.saxutils import escape as esc

    ss = getSampleStyleSheet()
    small = ParagraphStyle("small", parent=ss["BodyText"], fontSize=8.5, leading=11, spaceAfter=3)
    def P(value, style=small):
        return Paragraph(esc(str(value)).replace("\n", "<br/>"), style)
    doc = SimpleDocTemplate(path, pagesize=A4, leftMargin=1.7*cm, rightMargin=1.7*cm,
                            topMargin=1.6*cm, bottomMargin=1.6*cm)
    score_text = f"{summary['score']}%" if summary.get("score") is not None else "Not calculable (no assessed controls)"
    story = [Paragraph("ISO/IEC 27001:2022 Annex A Assessment", ss["Title"]),
             P(f"Organization: {company} | Date: {date.today()}"),
             P(f"Evidence-based score: {score_text} | Assessment coverage: {summary['coverage']}% ({summary['assessed']} of {summary['total']} controls assessed)"),
             P("Scoring method: Compliant = 1.0, Partially compliant = 0.5, Non-compliant = 0.0. No Evidence Found is excluded from the score denominator and reported separately."),
             P("Scope limitation: this result reflects only the supplied documents and retrieved evidence. Confirm control applicability against the organization's Statement of Applicability and risk treatment plan."),
             Spacer(1, 10)]
    rows = [["Status", "Count"]] + [[STATUS_LABEL[s], summary[s]] for s in STATUSES] + [["Total controls", summary["total"]]]
    table = Table(rows, colWidths=[9*cm, 3*cm], repeatRows=1)
    table.setStyle(TableStyle([("GRID", (0,0), (-1,-1), 0.4, colors.grey),
                               ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
                               ("VALIGN", (0,0), (-1,-1), "TOP")] ))
    story += [table, Spacer(1, 12)]
    for status in ["non_compliant", "partial", "no_evidence", "compliant"]:
        items = [r for r in results if r["status"] == status]
        if not items:
            continue
        story.append(Paragraph(f"{STATUS_LABEL[status]} ({len(items)})", ss["Heading2"]))
        for r in items:
            block = [Paragraph(esc(f"{r['id']} — {r['title']}"), ss["Heading4"]),
                     P("Requirement: " + r["requirement"]), P("Assessment: " + r["reasoning"])]
            for e in r.get("evidence", []):
                tag = f" [{e.get('tag')}]" if e.get("tag") else ""
                block.append(P(f"Source{tag}: {e['ref']} | Validation: {'validated citation' if e.get('verified') else 'unverified'} | {e['quote']}"))
            if r.get("gaps"):
                block.append(P("Gaps: " + "; ".join(str(g) for g in r["gaps"])))
            if r.get("contradictions"):
                block.append(P("Contradictions requiring resolution: " + "; ".join(str(x) for x in r["contradictions"])))
            if r.get("recommendation"):
                block.append(P("Recommended action: " + r["recommendation"]))
            block.append(Spacer(1, 7))
            story.append(KeepTogether(block))
    doc.build(story)
    return path

In [15]:
build_pdf(results, summary, "compliance_report.pdf", company="Northstar Digital Services (fictional test dossier)")
print("Saved compliance_report.pdf")

Saved compliance_report.pdf


## 🖥️ Dashboard (drag & drop)
Everything above is available as the `regulatory_core.py` module, which the dashboard uses:
```bash
streamlit run app.py
```
It has three tabs: **Upload documents** (drag and drop files), **Compliance audit** (chart, table, evidence, recommendations, PDF download) and **Ask the documents** (chat).